# 🌏 Goofy Screener — Phase 3: Multi-Market Autonomous Strategy Screener

**Markets:** 🇺🇸 US &nbsp;|&nbsp; 🇦🇺 ASX &nbsp;|&nbsp; 🇯🇵 Japan (JPX)

**Strategies tested per asset:**
| Strategy | Logic |
|---|---|
| MA Crossover | Short MA > Long MA → long |
| RSI | Buy oversold, sell overbought |
| Bollinger Bands | Buy below lower band, sell at midline |
| MACD | MACD line > signal line → long |
| Mean Reversion | Z-score < −threshold → long, exit at 0 |

---

**What this notebook does:**
1. Downloads live price data for ~120 stocks across all 3 markets
2. Runs a full parameter grid search on each strategy (train: 2016–2020)
3. Validates the best strategy out-of-sample (test: 2021–present)
4. Scores each asset 0–100 using Sharpe, Return, Max Drawdown, and DD Saved vs B&H
5. Ranks into tiers: **S ⭐ / A ✅ / B 🔵 / Skip ⬜**
6. Saves a colour-coded multi-tab Excel report

---
**Author:** Hiroki Kunu &nbsp;|&nbsp; University of Queensland &nbsp;|&nbsp; [GitHub: GoofyisDAWG](https://github.com/GoofyisDAWG)

## Cell 1 — Install & Import

In [1]:
# Run this cell first — installs required packages if missing
import subprocess, sys
for pkg in ['yfinance', 'openpyxl']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import yfinance as yf
import numpy as np
import pandas as pd
import datetime as dt
import os, warnings
warnings.filterwarnings('ignore')

try:
    from openpyxl import load_workbook, Workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    EXCEL_FORMAT = True
    print('✅ All packages loaded.')
except ImportError:
    EXCEL_FORMAT = False
    print('⚠️  openpyxl not found — plain Excel only.')

✅ All packages loaded.


## Cell 2 — Configuration

> **Change `MARKET` here to run only one market, or leave as `'ALL'` to run everything.**
> Options: `'ALL'` | `'US'` | `'ASX'` | `'JPX'`

In [2]:
# ── User settings — change these if needed ────────────────────────────────
MARKET       = 'ALL'   # 'ALL', 'US', 'ASX', or 'JPX'

# ── Date ranges ───────────────────────────────────────────────────────────────
TRAIN_START      = dt.datetime(2016, 1, 1)
TRAIN_END        = dt.datetime(2021, 1, 1)
TEST_END         = dt.datetime.now()
MIN_ROWS         = 400        # minimum trading days to include an asset
PERIODS_PER_YEAR = 252

# ── Tier thresholds (scoring 0–100) ───────────────────────────────────────────
# Scoring breakdown:
#   Sharpe ratio     → 40 pts max  (quality of risk-adjusted return)
#   Total return     → 25 pts max  (raw performance)
#   Max drawdown     → 20 pts max  (capital preservation)
#   DD saved vs B&H  → 15 pts max  (strategy value-add over buy & hold)
TIER_S = {'sharpe': 0.8,  'ret': 30,  'max_dd': -20}   # ⭐ Excellent
TIER_A = {'sharpe': 0.4,  'ret': 10,  'max_dd': -35}   # ✅ Good
TIER_B = {'sharpe': 0.1,  'ret': -10, 'max_dd': -50}   # 🔵 Decent
# Below TIER_B → Skip

# ── Output folder ─────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(os.getcwd(), 'screener_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

TODAY = dt.datetime.now().strftime('%Y-%m-%d')
print(f'Config loaded. Run date: {TODAY}')
print(f'Market selection: {MARKET}')
print(f'Train: {TRAIN_START.date()} → {TRAIN_END.date()}   |   Test: {TRAIN_END.date()} → {TEST_END.date()}')

Config loaded. Run date: 2026-04-14
Market selection: ALL
Train: 2016-01-01 → 2021-01-01   |   Test: 2021-01-01 → 2026-04-14


## Cell 3 — Stock Universes

| Market | Count | Coverage |
|--------|-------|----------|
| 🇺🇸 US  | 40    | Tech, Financials, Healthcare, Energy, Consumer, ETFs |
| 🇦🇺 ASX | 31    | Big 4 Banks, Resources, Healthcare, Retail, REITs, ETFs |
| 🇯🇵 JPX | 48    | Automotive, Electronics, Gaming, Financials, Telecom, Trading Cos |


In [3]:
# 🇺🇸 US Universe — Large-cap + sector representation + ETFs
US_UNIVERSE = [
    # Tech mega-cap
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AMZN', 'TSLA', 'AMD',
    # Financials
    'JPM', 'BAC', 'GS', 'MS', 'V', 'MA', 'BRK-B',
    # Healthcare
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV', 'MRK',
    # Energy
    'XOM', 'CVX', 'COP',
    # Consumer
    'PG', 'KO', 'PEP', 'WMT', 'COST', 'MCD',
    # Industrials
    'CAT', 'DE', 'BA', 'HON', 'RTX',
    # ETFs (benchmark)
    'SPY', 'QQQ', 'GLD', 'TLT', 'IWM',
]

# 🇦🇺 ASX Universe — ASX 200 sector coverage
ASX_UNIVERSE = [
    # Big 4 Banks
    'CBA.AX', 'WBC.AX', 'ANZ.AX', 'NAB.AX',
    # Resources — Iron Ore
    'BHP.AX', 'RIO.AX', 'FMG.AX', 'S32.AX',
    # Energy
    'WDS.AX', 'STO.AX', 'BPT.AX',
    # Healthcare
    'CSL.AX', 'RMD.AX', 'COH.AX', 'SHL.AX',
    # Retail / Consumer
    'WES.AX', 'WOW.AX', 'COL.AX', 'JBH.AX',
    # Tech
    'XRO.AX', 'WTC.AX', 'ALU.AX',
    # Infrastructure / Utilities
    'TCL.AX', 'APA.AX', 'SKI.AX',
    # REITs
    'GMG.AX', 'SCG.AX', 'GPT.AX',
    # ETFs (benchmark)
    'IOZ.AX', 'STW.AX', 'VAS.AX',
]

# 🇯🇵 JPX Universe — Nikkei 225 major components
JPX_UNIVERSE = [
    # Automotive
    '7203.T',  # Toyota Motor
    '7267.T',  # Honda Motor
    '7261.T',  # Mazda Motor
    '7272.T',  # Yamaha Motor
    '7269.T',  # Suzuki Motor
    # Electronics / Technology
    '6758.T',  # Sony Group
    '6501.T',  # Hitachi
    '6954.T',  # Fanuc
    '6902.T',  # Denso
    '6861.T',  # Keyence
    '6762.T',  # TDK
    '8035.T',  # Tokyo Electron
    '6857.T',  # Advantest
    '6723.T',  # Renesas Electronics
    '6594.T',  # Nidec
    # Semiconductors / Chemicals
    '4063.T',  # Shin-Etsu Chemical
    '4523.T',  # Eisai
    # Telecom
    '9432.T',  # NTT
    '9433.T',  # KDDI
    '9434.T',  # SoftBank Corp
    # Internet / Media
    '9984.T',  # SoftBank Group
    '4689.T',  # Z Holdings
    '6098.T',  # Recruit Holdings
    '4385.T',  # Mercari
    # Gaming / Entertainment
    '7974.T',  # Nintendo
    '9684.T',  # Square Enix
    '7832.T',  # Bandai Namco
    # Financials — Banks
    '8306.T',  # MUFG (Mitsubishi UFJ)
    '8316.T',  # Sumitomo Mitsui
    '8411.T',  # Mizuho Financial
    # Financials — Insurance
    '8750.T',  # Dai-ichi Life
    '8725.T',  # MS&AD Insurance
    # Consumer / Retail
    '3382.T',  # Seven & I Holdings
    '8267.T',  # Aeon
    '4661.T',  # Oriental Land (Disney Japan)
    '2914.T',  # Japan Tobacco
    # Industrials / Machinery
    '6301.T',  # Komatsu
    '6326.T',  # Kubota
    '7011.T',  # Mitsubishi Heavy Industries
    # Pharma / Healthcare
    '4502.T',  # Takeda Pharmaceutical
    '4519.T',  # Chugai Pharmaceutical
    # Trading Companies
    '8001.T',  # Itochu
    '8002.T',  # Marubeni
    '8058.T',  # Mitsubishi Corp
    # Transport / Infrastructure
    '9022.T',  # Central Japan Railway
    '9020.T',  # East Japan Railway
    # ETFs (benchmark)
    '1321.T',  # Nikkei 225 ETF (Nomura)
    '1306.T',  # TOPIX ETF (Nomura)
]

UNIVERSE_MAP = {'US': US_UNIVERSE, 'ASX': ASX_UNIVERSE, 'JPX': JPX_UNIVERSE}
MARKETS_TO_RUN = list(UNIVERSE_MAP.keys()) if MARKET == 'ALL' else [MARKET]

total = sum(len(UNIVERSE_MAP[m]) for m in MARKETS_TO_RUN)
print(f'Markets to run: {MARKETS_TO_RUN}')
print(f'Total assets in scope: {total}')

Markets to run: ['US', 'ASX', 'JPX']
Total assets in scope: 119


## Cell 4 — Metrics, Strategies & Parameter Grids

All 5 strategy functions + parameter grids for grid search. Same logic as the individual strategy notebooks.

In [4]:
# ── Performance metrics calculator ────────────────────────────────────────────
def compute_metrics(price, position):
    df = pd.DataFrame({'price': price})
    df['pos']       = position.reindex(df.index).fillna(0)
    df['log_ret']   = np.log(df['price'] / df['price'].shift(1))
    df['strat_ret'] = df['pos'] * df['log_ret']
    df['cum']       = np.exp(df['strat_ret'].cumsum())
    lr = df['strat_ret'].dropna()
    if len(lr) < 10:
        return {'Sharpe': np.nan, 'TotalRet': np.nan, 'MaxDD': np.nan, 'WinRate': np.nan}
    ann    = np.exp(lr.mean() * PERIODS_PER_YEAR) - 1
    vol    = lr.std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe = ann / vol if vol != 0 else np.nan
    cum    = df['cum'].dropna()
    mdd    = ((cum - cum.cummax()) / cum.cummax()).min() if len(cum) > 0 else np.nan
    total  = cum.iloc[-1] - 1 if len(cum) > 0 else np.nan
    wins   = (lr > 0).sum() / max(len(lr), 1)
    return {
        'Sharpe':   round(float(sharpe), 3) if not np.isnan(sharpe) else np.nan,
        'TotalRet': round(float(total) * 100, 2) if not np.isnan(total) else np.nan,
        'MaxDD':    round(float(mdd) * 100, 2) if not np.isnan(mdd) else np.nan,
        'WinRate':  round(float(wins) * 100, 1),
    }

# ── Strategy signal functions ──────────────────────────────────────────────────
def strategy_ma(p, short=20, long=50):
    return (p.rolling(short).mean() > p.rolling(long).mean()).astype(int).shift(1)

def strategy_rsi(p, period=14, oversold=30, overbought=70):
    d   = p.diff()
    rsi = 100 - (100 / (1 + d.clip(lower=0).rolling(period).mean() /
                            (-d.clip(upper=0)).rolling(period).mean()))
    sig = np.zeros(len(p)); hold = False
    for i, r in enumerate(rsi):
        if np.isnan(r): sig[i] = 0
        elif not hold and r < oversold:  hold = True;  sig[i] = 1
        elif hold and r > overbought:    hold = False; sig[i] = 0
        else: sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

def strategy_bb(p, window=20, num_std=2.0):
    mid = p.rolling(window).mean()
    low = mid - num_std * p.rolling(window).std()
    sig = np.zeros(len(p)); hold = False
    for i in range(len(p)):
        pi, m, l = p.iloc[i], mid.iloc[i], low.iloc[i]
        if np.isnan(l): sig[i] = 0
        elif not hold and pi <= l: hold = True;  sig[i] = 1
        elif hold and pi >= m:     hold = False; sig[i] = 0
        else: sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

def strategy_macd(p, fast=12, slow=26, signal_p=9):
    macd = p.ewm(span=fast).mean() - p.ewm(span=slow).mean()
    return (macd > macd.ewm(span=signal_p).mean()).astype(int).shift(1)

def strategy_mr(p, window=20, threshold=1.5):
    z   = (p - p.rolling(window).mean()) / p.rolling(window).std()
    sig = np.zeros(len(p)); hold = False
    for i, zi in enumerate(z):
        if np.isnan(zi): sig[i] = 0
        elif not hold and zi < -threshold: hold = True;  sig[i] = 1
        elif hold and zi >= 0:             hold = False; sig[i] = 0
        else: sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

STRATEGY_FNS = {
    'MA Crossover':    strategy_ma,
    'RSI':             strategy_rsi,
    'Bollinger Bands': strategy_bb,
    'MACD':            strategy_macd,
    'Mean Reversion':  strategy_mr,
}

STRATEGY_GRIDS = {
    'MA Crossover':    [{'short': s, 'long': l}
                        for s in [10, 20, 50] for l in [50, 100, 200] if s < l],
    'RSI':             [{'period': p, 'oversold': os, 'overbought': ob}
                        for p in [10, 14, 21] for os, ob in [(25,65),(30,70),(35,75)]],
    'Bollinger Bands': [{'window': w, 'num_std': s}
                        for w in [10, 20, 30] for s in [1.5, 2.0, 2.5]],
    'MACD':            [{'fast': f, 'slow': s, 'signal_p': sg}
                        for f in [8, 12] for s in [21, 26] for sg in [7, 9] if f < s],
    'Mean Reversion':  [{'window': w, 'threshold': t}
                        for w in [10, 20, 40] for t in [1.0, 1.5, 2.0]],
}

total_combos = sum(len(v) for v in STRATEGY_GRIDS.values())
print(f'✅ Strategies loaded: {list(STRATEGY_FNS.keys())}')
print(f'   Total parameter combinations per asset: {total_combos}')

✅ Strategies loaded: ['MA Crossover', 'RSI', 'Bollinger Bands', 'MACD', 'Mean Reversion']
   Total parameter combinations per asset: 43


## Cell 5 — Scoring & Tier Classification

Each asset gets a **composite score (0–100)** after out-of-sample testing:

| Component | Max Points | What it measures |
|---|---|---|
| OUT Sharpe ratio | 40 | Quality of risk-adjusted return |
| OUT Total Return | 25 | Raw performance (log-scaled) |
| Max Drawdown protection | 20 | Capital preservation |
| DD Saved vs Buy & Hold | 15 | Strategy value-add over just holding |

**Tier thresholds:**
- **S ⭐ Excellent:** Sharpe ≥ 0.8, Return ≥ 30%, Max DD ≥ −20%
- **A ✅ Good:** Sharpe ≥ 0.4, Return ≥ 10%, Max DD ≥ −35%
- **B 🔵 Decent:** Sharpe ≥ 0.1, Max DD ≥ −50%
- **Skip ⬜:** Below all thresholds

In [5]:
def score_asset(row):
    sharpe  = row.get('OUT Sharpe', np.nan)
    ret     = row.get('OUT Strat Ret %', np.nan)
    mdd     = row.get('OUT Strat Max DD %', np.nan)
    dd_save = row.get('DD Saved %', np.nan)
    if any(pd.isna(v) for v in [sharpe, ret, mdd]):
        return ('Skip', 0.0)
    s_pts  = min(max(sharpe / 1.5, 0), 1) * 40
    r_pts  = min(max(ret + 30, 0) / 130, 1) * 25
    d_pts  = min(max((60 + mdd) / 60, 0), 1) * 20
    ds_pts = min(max((dd_save + 20) / 60, 0), 1) * 15 if not pd.isna(dd_save) else 0
    score  = round(s_pts + r_pts + d_pts + ds_pts, 1)
    if sharpe >= TIER_S['sharpe'] and ret >= TIER_S['ret'] and mdd >= TIER_S['max_dd']:
        tier = 'S'
    elif sharpe >= TIER_A['sharpe'] and ret >= TIER_A['ret'] and mdd >= TIER_A['max_dd']:
        tier = 'A'
    elif sharpe >= TIER_B['sharpe'] and mdd >= TIER_B['max_dd']:
        tier = 'B'
    else:
        tier = 'Skip'
    return (tier, score)

print('✅ Scoring function ready.')

✅ Scoring function ready.


## Cell 6 — Screener & Excel Report Functions

In [6]:
TIER_COLORS  = {'S': 'FFD700', 'A': '90EE90', 'B': '87CEEB', 'Skip': 'D3D3D3'}
STRAT_COLORS = {
    'MA Crossover': 'AED6F1', 'RSI': 'FAD7A0',
    'Bollinger Bands': 'A9DFBF', 'MACD': 'D7BDE2', 'Mean Reversion': 'F1948A'
}

def screen_market(market_name, assets, price_data):
    results = []
    valid = [a for a in assets if a in price_data]
    print(f'\n  Screening {market_name} ({len(valid)} assets)...')
    for asset in valid:
        full  = price_data[asset]
        train = full[full.index < TRAIN_END]
        test  = full[full.index >= TRAIN_END]
        if len(train) < 100 or len(test) < 50:
            continue
        best_sharpe, best_strat, best_params = -999, None, None
        for strat_name, fn in STRATEGY_FNS.items():
            for params in STRATEGY_GRIDS[strat_name]:
                try:
                    m = compute_metrics(train, fn(train, **params))
                    if not np.isnan(m['Sharpe']) and m['Sharpe'] > best_sharpe:
                        best_sharpe, best_strat, best_params = m['Sharpe'], strat_name, params
                except: continue
        if best_strat is None: continue
        tm      = compute_metrics(test, STRATEGY_FNS[best_strat](test, **best_params))
        log     = np.log(test / test.shift(1)).dropna()
        bah_ret = (np.exp(log.cumsum()).iloc[-1] - 1) * 100 if len(log) > 0 else np.nan
        bah_cum = np.exp(log.cumsum()) if len(log) > 0 else pd.Series(dtype=float)
        bah_mdd = ((bah_cum - bah_cum.cummax()) / bah_cum.cummax()).min() * 100 if len(bah_cum) > 0 else np.nan
        dd_saved = round(tm['MaxDD'] - float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan
        row = {
            'Market': market_name, 'Asset': asset,
            'Best Strategy': best_strat, 'Best Params': str(best_params),
            'Train Sharpe': round(best_sharpe, 3),
            'OUT Sharpe': tm['Sharpe'], 'OUT Win Rate %': tm['WinRate'],
            'OUT Strat Ret %': tm['TotalRet'],
            'OUT B&H Ret %': round(float(bah_ret), 2) if not np.isnan(bah_ret) else np.nan,
            'OUT Strat Max DD %': tm['MaxDD'],
            'OUT B&H Max DD %': round(float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan,
            'DD Saved %': dd_saved,
            'Beats B&H': (tm['TotalRet'] or 0) > (bah_ret or 0),
            'Run Date': TODAY,
        }
        tier, score = score_asset(row)
        row['Tier'] = tier; row['Score'] = score
        results.append(row)
        icon = {'S':'⭐','A':'✅','B':'🔵','Skip':'⬜'}.get(tier,'')
        dd_str = f"{dd_saved:+.1f}%" if not np.isnan(dd_saved) else 'N/A'
        print(f'  {icon}[{tier}] {asset:14} | {best_strat:18} | '
              f'Sharpe:{tm["Sharpe"]:5.2f} | Ret:{(tm["TotalRet"] or 0):6.0f}% | '
              f'DD:{(tm["MaxDD"] or 0):5.1f}% | DDsaved:{dd_str:>8} | Score:{score:.0f}/100')
    return pd.DataFrame(results)


def apply_fmt(ws, df):
    if not EXCEL_FORMAT: return
    try:
        THIN = Side(style='thin', color='CCCCCC')
        BRD  = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
        HDR  = PatternFill('solid', fgColor='1C2833')
        GRN  = PatternFill('solid', fgColor='D5F5E3')
        RED  = PatternFill('solid', fgColor='FADBD8')
        hdrs = [c.value for c in ws[1]]
        ci   = {h: i+1 for i, h in enumerate(hdrs)}
        for cell in ws[1]:
            cell.fill=HDR; cell.font=Font(bold=True,color='FFFFFF',size=10)
            cell.alignment=Alignment(horizontal='center'); cell.border=BRD
        ws.row_dimensions[1].height = 28
        for row in ws.iter_rows(min_row=2):
            sv = row[ci['Best Strategy']-1].value if 'Best Strategy' in ci else None
            tv = row[ci['Tier']-1].value if 'Tier' in ci else None
            bv = row[ci['Beats B&H']-1].value if 'Beats B&H' in ci else None
            for cell in row:
                cell.border=BRD; cell.alignment=Alignment(horizontal='center'); cell.font=Font(size=10)
                h = hdrs[cell.column-1]
                if h=='Best Strategy' and sv: cell.fill=PatternFill('solid',fgColor=STRAT_COLORS.get(sv,'FFFFFF')); cell.font=Font(bold=True,size=10)
                elif h=='Tier' and tv: cell.fill=PatternFill('solid',fgColor=TIER_COLORS.get(tv,'FFFFFF')); cell.font=Font(bold=True,size=10)
                elif h=='Score':
                    try:
                        v=float(cell.value or 0)
                        if v>=70: cell.fill=PatternFill('solid',fgColor='27AE60')
                        elif v>=50: cell.fill=PatternFill('solid',fgColor='F39C12')
                        elif v>=30: cell.fill=PatternFill('solid',fgColor='AED6F1')
                    except: pass
                elif h=='OUT Sharpe':
                    try:
                        v=float(cell.value or 0)
                        if v>=0.8: cell.fill=GRN
                        elif v<0: cell.fill=RED
                    except: pass
                elif h in ('OUT Strat Max DD %','OUT B&H Max DD %'):
                    try:
                        v=float(cell.value or 0)
                        if v>=-20: cell.fill=GRN
                        elif v<-40: cell.fill=RED
                    except: pass
                elif h=='DD Saved %':
                    try:
                        v=float(cell.value or 0)
                        if v>=10: cell.fill=GRN
                        elif v<0: cell.fill=RED
                    except: pass
                elif h=='Beats B&H': cell.fill = GRN if bv else RED
        for col in ws.columns:
            w = max((len(str(c.value or '')) for c in col), default=0)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(w+4, 32)
        ws.freeze_panes = 'A2'
    except Exception as e: print(f'  [WARN] Formatting: {e}')


def save_excel(all_results, today):
    path = os.path.join(OUTPUT_DIR, f'Goofy_Phase3_{today}.xlsx')
    wb = Workbook(); wb.remove(wb.active)
    flags = {'US':'🇺🇸','ASX':'🇦🇺','JPX':'🇯🇵'}
    all_dfs = []
    for mkt, df in all_results.items():
        if df.empty: continue
        dfs = df.sort_values('Score', ascending=False).reset_index(drop=True)
        cols = [c for c in dfs.columns if c != 'Best Params']
        ws = wb.create_sheet(f"{flags.get(mkt,'')} {mkt}")
        ws.append(cols)
        for _, r in dfs[cols].iterrows(): ws.append([r[c] for c in cols])
        apply_fmt(ws, dfs[cols]); all_dfs.append(dfs)
    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)
        top = combined[combined['Tier'].isin(['S','A'])].sort_values('Score',ascending=False)
        decent = combined[combined['Tier']=='B'].sort_values('Score',ascending=False)
        ws0 = wb.create_sheet('⭐ Top Performers', 0)
        ws0.append(['GOOFY SCREENER — PHASE 3'])
        ws0.append([f'Run: {today}  |  Markets: US + ASX + JPX  |  Assets: {len(combined)}'])
        ws0.append([])
        ws0.append(['TIER','CRITERIA'])
        for row in [['S ⭐','Sharpe≥0.8, Ret≥30%, MaxDD≥-20%'],
                    ['A ✅','Sharpe≥0.4, Ret≥10%, MaxDD≥-35%'],
                    ['B 🔵','Sharpe≥0.1, MaxDD≥-50%'],
                    ['Skip ⬜','Below all thresholds']]: ws0.append(row)
        ws0.append([])
        ws0.append(['TIER BREAKDOWN'])
        for t in ['S','A','B','Skip']: ws0.append([t, len(combined[combined['Tier']==t])])
        ws0.append([])
        ws0.append(['── S & A TIER (Best Opportunities) ──'])
        cols_show = ['Market','Asset','Best Strategy','Tier','Score',
                     'OUT Sharpe','OUT Strat Ret %','OUT B&H Ret %','OUT Strat Max DD %','DD Saved %','Beats B&H']
        ws0.append(cols_show)
        for _, r in top[cols_show].iterrows(): ws0.append([r[c] for c in cols_show])
        ws0.append([])
        ws0.append(['── B TIER (Decent — worth watching) ──'])
        cols_b = ['Market','Asset','Best Strategy','Score','OUT Sharpe','OUT Strat Ret %','OUT Strat Max DD %','DD Saved %']
        ws0.append(cols_b)
        for _, r in decent[cols_b].iterrows(): ws0.append([r[c] for c in cols_b])
        if EXCEL_FORMAT:
            try: ws0['A1'].font=Font(bold=True,size=14,color='1C2833')
            except: pass
        ws_d = wb.create_sheet('📊 Strategy Distribution')
        ws_d.append(['Strategy','US','ASX','JPX','Total'])
        for s in STRATEGY_FNS:
            row = [s]
            tot = 0
            for m in ['US','ASX','JPX']:
                c = (all_results[m]['Best Strategy']==s).sum() if m in all_results and not all_results[m].empty else 0
                row.append(c); tot+=c
            row.append(tot); ws_d.append(row)
    wb.save(path)
    return path

print('✅ Screener and Excel functions ready.')

✅ Screener and Excel functions ready.


## Cell 7 — Step 1: Download Price Data

> ⏱️ **This cell takes 2–5 minutes** depending on internet speed. Each tick mark = 1 asset downloaded successfully.

In [7]:
# Build full asset list (deduplicated)
all_assets = []
for m in MARKETS_TO_RUN:
    all_assets.extend(UNIVERSE_MAP[m])
seen = set(); unique_assets = []
for a in all_assets:
    if a not in seen: unique_assets.append(a); seen.add(a)

print(f'Downloading {len(unique_assets)} assets...')
print(f'Train: {TRAIN_START.date()}  →  Test end: {TEST_END.date()}\n')

price_data = {}
failed = []
for i, asset in enumerate(unique_assets):
    try:
        raw = yf.download(asset, start=TRAIN_START, end=TEST_END,
                          auto_adjust=True, progress=False)
        if not raw.empty and len(raw) >= MIN_ROWS:
            close = raw['Close'].squeeze()
            if isinstance(close, pd.DataFrame): close = close.iloc[:, 0]
            price_data[asset] = close
            print(f'  ✓ {asset:14} | {len(raw)} rows', end='\n')
        else:
            failed.append(asset)
            print(f'  ✗ {asset:14} | skipped ({len(raw)} rows < {MIN_ROWS})')
    except Exception as e:
        failed.append(asset)
        print(f'  ✗ {asset:14} | error: {e}')

print(f'\n✅ {len(price_data)} assets ready   |   {len(failed)} skipped')
if failed:
    print(f'   Skipped: {failed}')

Train: 2016-01-01  →  Test end: 2026-04-14

  ✓ AAPL           | 2583 rows
  ✓ MSFT           | 2583 rows
  ✓ NVDA           | 2583 rows
  ✓ GOOGL          | 2583 rows
  ✓ META           | 2583 rows
  ✓ AMZN           | 2583 rows
  ✓ TSLA           | 2583 rows
  ✓ AMD            | 2583 rows
  ✓ JPM            | 2583 rows
  ✓ BAC            | 2583 rows
  ✓ GS             | 2583 rows
  ✓ MS             | 2583 rows
  ✓ V              | 2583 rows
  ✓ MA             | 2583 rows
  ✓ BRK-B          | 2583 rows
  ✓ JNJ            | 2583 rows
  ✓ UNH            | 2583 rows
  ✓ LLY            | 2583 rows
  ✓ PFE            | 2583 rows
  ✓ ABBV           | 2583 rows
  ✓ MRK            | 2583 rows
  ✓ XOM            | 2583 rows
  ✓ CVX            | 2583 rows
  ✓ COP            | 2583 rows
  ✓ PG             | 2583 rows
  ✓ KO             | 2583 rows
  ✓ PEP            | 2583 rows
  ✓ WMT            | 2583 rows
  ✓ COST           | 2583 rows
  ✓ MCD            | 2583 rows
  ✓ CAT            | 2583 

$ALU.AX: possibly delisted; no timezone found

1 Failed download:
['ALU.AX']: possibly delisted; no timezone found


  ✗ ALU.AX         | skipped (0 rows < 400)
  ✓ TCL.AX         | 2599 rows
  ✓ APA.AX         | 2599 rows


$SKI.AX: possibly delisted; no timezone found

1 Failed download:
['SKI.AX']: possibly delisted; no timezone found


  ✗ SKI.AX         | skipped (0 rows < 400)
  ✓ GMG.AX         | 2599 rows
  ✓ SCG.AX         | 2599 rows
  ✓ GPT.AX         | 2599 rows
  ✓ IOZ.AX         | 2598 rows
  ✓ STW.AX         | 2598 rows
  ✓ VAS.AX         | 2598 rows
  ✓ 7203.T         | 2532 rows
  ✓ 7267.T         | 2532 rows
  ✓ 7261.T         | 2532 rows
  ✓ 7272.T         | 2532 rows
  ✓ 7269.T         | 2532 rows
  ✓ 6758.T         | 2532 rows
  ✓ 6501.T         | 2532 rows
  ✓ 6954.T         | 2532 rows
  ✓ 6902.T         | 2532 rows
  ✓ 6861.T         | 2532 rows
  ✓ 6762.T         | 2532 rows
  ✓ 8035.T         | 2532 rows
  ✓ 6857.T         | 2532 rows
  ✓ 6723.T         | 2532 rows
  ✓ 6594.T         | 2532 rows
  ✓ 4063.T         | 2532 rows
  ✓ 4523.T         | 2532 rows
  ✓ 9432.T         | 2532 rows
  ✓ 9433.T         | 2532 rows
  ✓ 9434.T         | 1782 rows
  ✓ 9984.T         | 2532 rows
  ✓ 4689.T         | 2532 rows
  ✓ 6098.T         | 2532 rows
  ✓ 4385.T         | 1913 rows
  ✓ 7974.T         | 2532 

## Cell 8 — Step 2: Run Strategy Screening

> ⏱️ **This cell takes 5–10 minutes** — it runs a full grid search across all strategies and parameters for every asset, then validates out-of-sample.

In [8]:
print('Running strategy screening...\n')
print(f'Each asset: {sum(len(v) for v in STRATEGY_GRIDS.values())} param combinations × 5 strategies')
print('=' * 75)

all_results = {}
for m in MARKETS_TO_RUN:
    all_results[m] = screen_market(m, UNIVERSE_MAP[m], price_data)

print('\n' + '=' * 75)
print('✅ Screening complete.')

Running strategy screening...

Each asset: 43 param combinations × 5 strategies

  Screening US (40 assets)...
  ✅[A] AAPL           | MACD               | Sharpe: 0.56 | Ret:    63% | DD:-20.9% | DDsaved:  +12.5% | Score:54/100
  🔵[B] MSFT           | MA Crossover       | Sharpe: 0.37 | Ret:    37% | DD:-21.2% | DDsaved:  +16.0% | Score:45/100
  ⬜[Skip] NVDA           | MA Crossover       | Sharpe: 0.94 | Ret:   374% | DD:-63.6% | DDsaved:   +2.7% | Score:56/100
  ⬜[Skip] GOOGL          | RSI                | Sharpe: 0.09 | Ret:    12% | DD:-44.3% | DDsaved:   +0.0% | Score:21/100
  ⬜[Skip] META           | RSI                | Sharpe:-0.10 | Ret:   -16% | DD:-71.5% | DDsaved:   +5.3% | Score:9/100
  🔵[B] AMZN           | MA Crossover       | Sharpe: 0.17 | Ret:    21% | DD:-27.4% | DDsaved:  +28.8% | Score:37/100
  🔵[B] TSLA           | MACD               | Sharpe: 0.80 | Ret:   326% | DD:-38.8% | DDsaved:  +34.8% | Score:67/100
  ⬜[Skip] AMD            | MA Crossover       | Sharpe:

## Cell 9 — Step 3: Results Summary

In [9]:
all_dfs = [df for df in all_results.values() if not df.empty]
if not all_dfs:
    print('No results — check data download.')
else:
    combined = pd.concat(all_dfs, ignore_index=True)

    print(f'\n{"="*70}')
    print(f'  GOOFY SCREENER — PHASE 3  |  {TODAY}')
    print(f'{"="*70}')
    print(f'  Total assets screened: {len(combined)}')
    for m in MARKETS_TO_RUN:
        df_m = all_results.get(m, pd.DataFrame())
        if not df_m.empty: print(f'    {m}: {len(df_m)} assets')

    print(f'\n  Tier Breakdown:')
    for tier in ['S', 'A', 'B', 'Skip']:
        icon  = {'S':'⭐','A':'✅','B':'🔵','Skip':'⬜'}[tier]
        count = len(combined[combined['Tier']==tier])
        bar   = '█' * count
        print(f'    {icon} {tier:5}: {count:3}  {bar}')

    for tier_name, tier_label in [('S','⭐ S-Tier (Excellent)'), ('A','✅ A-Tier (Good)')]:
        print(f'\n  {tier_label}:')
        subset = combined[combined['Tier']==tier_name].sort_values('Score', ascending=False)
        if subset.empty: print('    (none this run)')
        else:
            for _, r in subset.iterrows():
                print(f'    {r["Market"]:5} {r["Asset"]:14} | {r["Best Strategy"]:18} | '
                      f'Sharpe:{r["OUT Sharpe"]:5.2f} | Ret:{r["OUT Strat Ret %"]:6.1f}% | '
                      f'DD:{r["OUT Strat Max DD %"]:5.1f}% | Score:{r["Score"]:.0f}/100')

    print(f'\n  Top 5 by Composite Score:')
    top5 = combined.nlargest(5, 'Score')[['Market','Asset','Best Strategy','Tier','Score',
                                          'OUT Sharpe','OUT Strat Ret %','OUT Strat Max DD %']]
    display(top5.reset_index(drop=True))

    print(f'\n  Top 5 DD Protectors (strategy saved most drawdown vs just holding):')
    top_dd = combined.nlargest(5, 'DD Saved %')[['Market','Asset','Best Strategy',
                                                  'OUT Strat Max DD %','OUT B&H Max DD %','DD Saved %']]
    display(top_dd.reset_index(drop=True))

    print(f'\n  Strategy Distribution:')
    for strat, count in combined['Best Strategy'].value_counts().items():
        pct = count / len(combined) * 100
        bar = '█' * int(pct/5)
        print(f'    {strat:20}: {count:3}  ({pct:.0f}%)  {bar}')


  GOOFY SCREENER — PHASE 3  |  2026-04-14
  Total assets screened: 117
    US: 40 assets
    ASX: 29 assets
    JPX: 48 assets

  Tier Breakdown:
    ⭐ S    :  12  ████████████
    ✅ A    :  30  ██████████████████████████████
    🔵 B    :  36  ████████████████████████████████████
    ⬜ Skip :  39  ███████████████████████████████████████

  ⭐ S-Tier (Excellent):
    JPX   8725.T         | RSI                | Sharpe: 1.31 | Ret: 110.1% | DD: -8.8% | Score:88/100
    JPX   8411.T         | MACD               | Sharpe: 1.07 | Ret: 156.4% | DD:-17.7% | Score:76/100
    JPX   8750.T         | MACD               | Sharpe: 1.09 | Ret: 189.1% | DD:-16.9% | Score:76/100
    JPX   8306.T         | MACD               | Sharpe: 1.09 | Ret: 167.5% | DD:-19.7% | Score:76/100
    ASX   NAB.AX         | MACD               | Sharpe: 0.93 | Ret:  82.8% | DD:-12.1% | Score:70/100
    JPX   9433.T         | Bollinger Bands    | Sharpe: 1.04 | Ret:  42.3% | DD: -6.4% | Score:67/100
    ASX   CBA.AX       

,Market,Asset,Best Strategy,Tier,Score,OUT Sharpe,OUT Strat Ret %,OUT Strat Max DD %
0,JPX,8725.T,RSI,S,87.8,1.306,110.06,-8.81
1,JPX,8411.T,MACD,S,76.4,1.066,156.37,-17.74
2,JPX,8750.T,MACD,S,76.4,1.088,189.08,-16.87
3,JPX,8306.T,MACD,S,75.6,1.094,167.46,-19.71
4,US,COP,RSI,A,75.0,1.120,247.52,-23.94



  Top 5 DD Protectors (strategy saved most drawdown vs just holding):


,Market,Asset,Best Strategy,OUT Strat Max DD %,OUT B&H Max DD %,DD Saved %
0,US,TSLA,MACD,-38.84,-73.63,34.79
1,ASX,CSL.AX,Bollinger Bands,-20.97,-55.13,34.16
2,US,GS,Bollinger Bands,0.00,-32.84,32.84
3,JPX,6594.T,MACD,-42.12,-73.93,31.81
4,ASX,APA.AX,Bollinger Bands,-7.91,-38.70,30.79



  Strategy Distribution:
    MA Crossover        :  34  (29%)  █████
    MACD                :  27  (23%)  ████
    RSI                 :  24  (21%)  ████
    Bollinger Bands     :  22  (19%)  ███
    Mean Reversion      :  10  (9%)  █


## Cell 10 — Full Results Table (sortable)

In [10]:
if 'combined' in dir():
    display_cols = ['Market','Asset','Best Strategy','Tier','Score',
                    'OUT Sharpe','OUT Strat Ret %','OUT B&H Ret %',
                    'OUT Strat Max DD %','OUT B&H Max DD %','DD Saved %','Beats B&H']
    full_table = combined[display_cols].sort_values('Score', ascending=False).reset_index(drop=True)
    display(full_table)
    print(f'\n{len(full_table)} assets total')

,Market,Asset,Best Strategy,Tier,Score,OUT Sharpe,OUT Strat Ret %,OUT B&H Ret %,OUT Strat Max DD %,OUT B&H Max DD %,DD Saved %,Beats B&H
0,JPX,8725.T,RSI,S,87.8,1.306,110.06,404.32,-8.81,-32.62,23.81,False
1,JPX,8750.T,MACD,S,76.4,1.088,189.08,369.11,-16.87,-29.08,12.21,False
2,JPX,8411.T,MACD,S,76.4,1.066,156.37,546.13,-17.74,-33.45,15.71,False
3,JPX,8306.T,MACD,S,75.6,1.094,167.46,670.37,-19.71,-31.85,12.14,False
4,US,COP,RSI,A,75.0,1.120,247.52,276.95,-23.94,-36.30,12.36,False
...,...,...,...,...,...,...,...,...,...,...,...,...
112,JPX,1306.T,RSI,Skip,5.1,-0.227,-74.59,-76.89,-90.49,-91.01,0.52,True
113,JPX,4523.T,RSI,Skip,4.3,-0.437,-46.84,-21.77,-72.13,-69.53,-2.60,False
114,US,MA,MA Crossover,Skip,3.9,-0.359,-28.40,49.33,-49.25,-28.25,-21.00,False
115,US,GS,Bollinger Bands,Skip,0.0,NaN,0.00,280.54,0.00,-32.84,32.84,False



117 assets total


## Cell 11 — Save Excel Report

In [11]:
if 'combined' in dir():
    xlsx_path = save_excel(all_results, TODAY)
    print(f'\n✅ Report saved!')
    print(f'   File: Goofy_Phase3_{TODAY}.xlsx')
    print(f'   Location: {OUTPUT_DIR}')
    print(f'\n   Open the screener_output/ folder to find your Excel report.')
    print(f'   Tabs: ⭐ Top Performers | 🇺🇸 US | 🇦🇺 ASX | 🇯🇵 JPX | 📊 Strategy Distribution')
else:
    print('Run cells 7–9 first to generate results.')


✅ Report saved!
   File: Goofy_Phase3_2026-04-14.xlsx
   Location: /var/www/filebrowser/.projects/66afd94c-b175-415a-8c90-55bfe9aa667f/screener_output

   Open the screener_output/ folder to find your Excel report.
   Tabs: ⭐ Top Performers | 🇺🇸 US | 🇦🇺 ASX | 🇯🇵 JPX | 📊 Strategy Distribution
